# Hướng Dẫn Giải Thích Chi Tiết: `src/features.py`

Notebook này phân tích cấu trúc, công dụng và cách hoạt động của lớp biến đổi dữ liệu `DataTransformer` trong `src/features.py`.

---

## 🔍 1. Tại Sao Cần Biến Đổi Dữ Liệu Thời Gian (Time-Series)?

Mô hình học máy truyền thống như hồi quy tuyến tính nhận đầu vào dạng bảng 2D `[số mẫu, số đặc trưng]`. Tuy nhiên, để các mô hình mạng Deep Learning (như Transformer) học được mối quan hệ tuần tự của giá cổ phiếu qua nhiều ngày liên tiếp, chúng ta cần biến đổi dữ liệu thành cấu trúc **3D**:
$$\text{Shape: } [\text{Số mẫu}, \text{Số bước thời gian (Lookback Window)}, \text{Số đặc trưng}]$$

Đồng thời, giá trị của các chỉ báo kỹ thuật có khoảng biến thiên rất khác nhau (ví dụ: RSI từ 0 đến 100, Khối lượng giao dịch từ hàng chục nghìn đến hàng triệu). Do đó, việc chuẩn hóa dữ liệu sử dụng **StandardScaler** (đưa dữ liệu về dạng có giá trị trung bình = 0 và độ lệch chuẩn = 1) là bắt buộc để mô hình tối ưu hội tụ nhanh.

In [1]:
import sys
import os
import numpy as np
import pandas as pd

# Thêm thư mục gốc vào đường dẫn hệ thống để import src
sys.path.append(os.path.abspath('..'))

from src.data_loader import fetch_and_prepare_data
from src.features import DataTransformer
print("Import lớp DataTransformer thành công!")

Import lớp DataTransformer thành công!


## ⚙️ 2. Các Bước Xử Lý Của Lớp `DataTransformer`

### Khởi tạo đối tượng
- `time_steps`: Kích thước cửa sổ trượt (lookback window), hiện tại được nâng cấp lên **45 ngày**.
- `feature_scaler`: Đối tượng `StandardScaler()` chuẩn hóa đầu vào.
- `target_scaler`: Đối tượng `StandardScaler()` chuẩn hóa đầu ra.

In [2]:
# Khởi tạo transformer
transformer = DataTransformer(time_steps=45)
print(f"Lookback window (time_steps): {transformer.time_steps}")
print(f"Danh sách các cột đặc trưng đầu vào ({len(transformer.feature_cols)} cột):")
print(transformer.feature_cols)

Lookback window (time_steps): 45
Danh sách các cột đặc trưng đầu vào (42 cột):
['gap_open', 'open_return', 'buying_pressure', 'shadow_ratio', 'intraday_range', 'return_1d', 'return_2d', 'return_3d', 'mom_5d', 'mom_10d', 'mom_20d', 'dist_ma50', 'volume_change', 'volume_sma_ratio', 'volume_zscore', 'ad_line_ratio', 'obv_zscore', 'vol_ratio', 'rsi_14', 'macd_ratio', 'bb_position', 'adx_14', 'stoch_k', 'efficiency_ratio', 'vix_lag1', 'bond_yield_lag1', 'usdvnd_change', 'vnindex_return_lag1', 'day_of_week_sin', 'day_of_week_cos', 'month_sin', 'month_cos', 'is_quarter_end', 'days_before_tet', 'mfi_14', 'dividend_flag', 'days_to_dividend', 'days_after_dividend', 'foreign_net_buy_proxy', 'foreign_net_buy_5d', 'foreign_net_buy_20d', 'self_net_buy_proxy']


### Hàm `fit_transform_data(df)`
Hàm này tính toán:
1. **Biến mục tiêu (Target):** Lợi suất mở cửa kế tiếp so với giá đóng cửa hôm nay:
   $$\text{Target}_t = \frac{\text{Open}_{t} - \text{Close}_{t-1}}{\text{Close}_{t-1}}$$
2. **Đặc trưng (Features):** Trích xuất 42 đặc trưng bao gồm các chỉ báo kỹ thuật, các biến trễ thời gian, tỷ suất sinh lời thị trường vĩ mô (`market_return`), chỉ số hoảng sợ `vix`, và đặc trưng cảm xúc tin tức (`sentiment_score`, `news_volume`) (chuẩn hóa StandardScaler), chuẩn hóa tất cả sử dụng **StandardScaler** (đưa dữ liệu về dạng có giá trị trung bình = 0 và độ lệch chuẩn = 1), giúp giảm thiểu tác động của giá trị ngoại lai.

In [3]:
# Tải dữ liệu thật
df_real = fetch_and_prepare_data("VNM.VN", start_date="2024-01-01", end_date="2024-12-31")

# Áp dụng fit_transform_data
X_scaled, y_scaled, y_spread_scaled = transformer.fit_transform_data(df_real)
print(f"Đặc trưng sau chuẩn hóa (shape): {X_scaled.shape}")
print(f"Giá trị đặc trưng lớn nhất: {X_scaled.max():,.2f}; Nhỏ nhất: {X_scaled.min():,.2f}")
print(f"Biến mục tiêu sau chuẩn hóa (shape): {y_scaled.shape}")

Đang tải tỷ giá USD/VND (USDVND=X) từ 2024-01-01 đến 2024-12-31...


Đang tải Yahoo Finance: VNM.VN...


[*********************100%***********************]  1 of 1 completed

  Yahoo Finance: 254 phiên
Đang nạp dữ liệu trường: C:\Users\ACER\Documents\Stock-Opening-Price-Prediction\data\raw\VNM_prices.csv
  Không tải được DNSE: object of type 'NoneType' has no len()
  Tổng: 1621 phiên (2019-09-17 → 2026-03-16)
Tính macro features...


  [NEWS] Tải và phân tích cảm xúc tin tức cho VNM.VN...


📥 Đang tải dữ liệu cổ tức cho VNM.VN từ yfinance...


   => Đã tải 3 đợt chia cổ tức cho VNM.VN
Đã lưu cache: C:\Users\ACER\Documents\Stock-Opening-Price-Prediction\data\VNM.VN_processed.csv
Sẵn sàng với 1617 phiên.



Đặc trưng sau chuẩn hóa (shape): (1617, 42)
Giá trị đặc trưng lớn nhất: 40.20; Nhỏ nhất: -40.20
Biến mục tiêu sau chuẩn hóa (shape): (1617, 3)


### Hàm `create_sliding_windows(X_data, y_data)`
Hàm này quét qua dữ liệu dạng bảng 2D và gom các nhóm gồm 45 phiên liên tiếp tạo thành cấu trúc 3D.
Ví dụ, nếu chúng ta muốn dự báo ngày thứ 46, đầu vào sẽ là thông tin từ ngày 1 đến ngày 45.

In [4]:
# Tạo sliding window dạng 3D
X_3D, y_3D, y_spread_3D = transformer.create_sliding_windows(X_scaled, y_scaled, y_spread_scaled)
print(f"Hình dạng dữ liệu 3D (X_3D): {X_3D.shape}")
print(f"Hình dạng mục tiêu tương ứng (y_3D): {y_3D.shape}")
print(f"Giải thích: Có {X_3D.shape[0]} mẫu dự báo, mỗi mẫu là một chuỗi gồm {X_3D.shape[1]} ngày liên tiếp, mỗi ngày chứa {X_3D.shape[2]} đặc trưng kỹ thuật và vĩ mô.")

Hình dạng dữ liệu 3D (X_3D): (1572, 45, 42)
Hình dạng mục tiêu tương ứng (y_3D): (1572, 3)
Giải thích: Có 1572 mẫu dự báo, mỗi mẫu là một chuỗi gồm 45 ngày liên tiếp, mỗi ngày chứa 42 đặc trưng kỹ thuật và vĩ mô.


### Hàm `split_train_test_chronological(df, X_3D, y_3D, train_ratio=0.8)`

**QUAN TRỌNG:** Trong dữ liệu chuỗi thời gian, ta **không bao giờ** được phân chia tập huấn luyện (train) và kiểm thử (test) ngẫu nhiên (như dùng `train_test_split` của Sklearn). Làm như vậy sẽ gây ra hiện tượng **rò rỉ dữ liệu tương lai** (Data Leakage) vào quá khứ.

Hàm này thực hiện chia cắt tuần tự theo thời gian thực tế: lấy 80% thời gian đầu để huấn luyện và 20% thời gian sau để kiểm thử.

In [5]:
X_train, y_train, X_test, y_test, y_test_raw, y_train_spread, y_test_spread = transformer.split_train_test_chronological(
    df_real, X_3D, y_3D, y_spread_3D=y_spread_3D, train_ratio=0.8
)

print(f"Số lượng mẫu tập Train: {X_train.shape[0]}")
print(f"Số lượng mẫu tập Test : {X_test.shape[0]}")

📊 Split 80/20 (Purge Gap: 45):
   🔹 Train: 1257 mẫu
   🔸 Test : 270 mẫu
Số lượng mẫu tập Train: 1257
Số lượng mẫu tập Test : 270
